# Round 20 · Pace and shot-variance context

**Prepared, not executed.** Read ROUND_20_PROTOCOL.md. Two symmetric game-context multipliers interact with existing strength and consensus gaps. Four differences in pace/shot profiles are the information control. No rating models are fitted.

Four candidates, two families, four controls, six fixed configurations. All 2022–2025 seasons were previously used; no untouched-test or leaderboard claim. GitHub authentication is not needed for this notebook.

In [ ]:
from pathlib import Path
import json,sys
import pandas as pd
import plotly.io as pio
from IPython.display import display,FileLink
KIT=Path.cwd().resolve()
if not (KIT/'run_round.py').is_file(): KIT=Path.home()/'march_next_steps'/'rounds_20_21'
assert (KIT/'run_round.py').is_file(), 'Open this notebook from march_next_steps/rounds_20_21.'
sys.path.insert(0,str(KIT))
from run_round import run_stage
from round_plots import figures
ROUND='20'
pio.renderers.default='plotly_mimetype'
assert json.loads((KIT/'reports/test_receipt.json').read_text())['status']=='PASS', 'Run run_tests.py first.'
print('Kernel:',sys.executable,'\nRound:',ROUND,'\nResearch folder:',KIT)
for r in ['17','18','19']:
    print('Previous round',r,json.loads((KIT/'evidence'/('round'+r)/'decisions.json').read_text()))

## 1. All-season data audit — zero fitting
Checks all twelve legal seasons before construction. Raw inputs and upstream receipts are immutable. A missing or different input stops; no download or rebuild is automatic.

In [ ]:
run_stage(ROUND,'audit',max_seconds=120)
RUN=Path(json.loads((KIT/'reports'/('round'+ROUND)/'latest_run.json').read_text())['run_dir'])
display(pd.read_csv(RUN/'data_quality.csv'))
assert json.loads((RUN/'quality_audit.json').read_text())['status']=='PASS'

## 2. Bounded 2013 smoke
Build only one training-season feature table. A technical pass is not an improvement result.

In [ ]:
run_stage(ROUND,'smoke',max_seconds=90)
print(json.dumps(json.loads((RUN/'smoke.json').read_text()),indent=2))

## 3. Immediately repeat the smoke to prove reuse
No new feature table or model fit should be needed.

In [ ]:
run_stage(ROUND,'smoke',max_seconds=90)
s=json.loads((RUN/'smoke.json').read_text())
assert s['new_feature_snapshots']==0 and s['new_rating_fits']==0
print('READY_FOR_REMAINING_PREPARATION')

## 4. Build remaining snapshots
Reuses 2013, creates at most eleven more snapshots, replays four existing reference classifiers. Zero new rating fits. Response/scoring proxies are descriptive, not calibrated win probabilities.

In [ ]:
run_stage(ROUND,'prepare',max_seconds=300)
print(json.dumps(json.loads((RUN/'prepare.json').read_text()),indent=2))
display(pd.read_csv(RUN/'coverage.csv'))
display(pd.read_csv(RUN/'feature_registry.csv'))

## 5. Fixed classifier comparison
Six configurations × four validation seasons. Four references replayed; at most twenty new classifier fits. Primary: both context families beyond descriptive controls. Duplicate-control comparison diagnoses regularization sensitivity, not independent information.

In [ ]:
run_stage(ROUND,'evaluate',max_seconds=180)
display(pd.read_csv(RUN/'metrics.csv')[['Season','recipe','brier','log_loss','source']].round(7))
display(pd.read_csv(RUN/'ablations.csv').round(7))
print(json.dumps(json.loads((RUN/'decisions.json').read_text()),indent=2))

## 6. Save scientific report before rendering
Aggregate return files exclude raw rows, trained models, individual predictions and team profiles. A negative scientific gate still produces a report.

In [ ]:
run_stage(ROUND,'report',max_seconds=120)
record=json.loads((KIT/'reports'/('round'+ROUND)/'latest_report.json').read_text())
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))

## 7. Inline Plotly evidence
Ten charts: previous findings, support, profiles, fixed scores, primary effect, duplicate sensitivity, family effects, train-only overlap, calibration and coefficient stability. Calibration bins and coefficient comparisons are descriptive, not causal.

In [ ]:
plots=figures(RUN,ROUND,KIT/'evidence')
assert len(plots)==10
for fig in plots[:5]: fig.show()

In [ ]:
for fig in plots[5:]: fig.show()

## Save and reopen
Press Ctrl+S, close and reopen to verify inline outputs remain. Shut down the kernel before the other round. Do not tune formulas or gates against the other round’s results. Missing prior round 16 is recorded separately; these two experiments do not depend on its fitted artifacts. After both complete, run package_returns.py.